# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow to load, explore, and process the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [mlcroissant](https://mlcommons.org/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Schema URL:**

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This loads the dataset's metadata and provides summary information so you can discover available record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
meta = dataset.metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}\nVersion: {meta.version}\nPublished: {getattr(meta, 'datePublished', 'unknown')}")

## 2. Data Overview
Review available **record sets** and their `@id`s, plus the fields and columns for each. Use this section to identify which data to extract in subsequent steps.

In [ ]:
# List record sets and their fields using @id
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- Record Set: name={record_set.name}, @id={record_set.id_}")
    if hasattr(record_set, 'fields'):
        print("  Fields in this Record Set:")
        for field in record_set.fields:
            colinfo = f" (from column @id={field.column.id_})" if hasattr(field, 'column') and hasattr(field.column, 'id_') else ''
            dtype = getattr(field, 'data_type', 'unknown')
            print(f"    - {field.name} (field @id={field.id_}, type={dtype}){colinfo}")
        print()
    else:
        print("  [No fields declared]")

## 3. Data Extraction
Load the main data record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. For this dataset, the main record set holding the protein analysis data can usually be found with a name such as "Protein Table" or similar. The code below selects the main table automatically.

In [ ]:
# Select the main record set (by inspecting for the table with the most fields)
if len(record_sets) == 0:
    raise ValueError("No record sets were found in the dataset.")
# Use the record set with the most fields (main table)
rs_main = max(record_sets, key=lambda rs: len(getattr(rs, 'fields', [])))
print(f"Selected main record set: name={rs_main.name}, @id={rs_main.id_}")

# Get the @id of the main record set
main_record_set_id = rs_main.id_

# List all record sets ids for demonstration
record_sets_ids = [rs.id_ for rs in record_sets]

# Extract all dataframes for each record set
dataframes = {}
for rs in record_sets:
    print(f"Loading records from record set: {rs.name} (@id={rs.id_}) ...")
    records = list(dataset.records(record_set=rs.id_))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id_] = df
        print(f"  --> Loaded {len(df)} records, columns: {df.columns.tolist()}")
    else:
        print("  --> No records found in this set.")

# Display columns and first few rows for the main record set
if main_record_set_id in dataframes:
    print(f"\nColumns in main record set (@id={main_record_set_id}): {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering by numeric fields, normalization, and simple grouping. We use the `@id` for each field.**

**You may need to refer to the printed column names above to select the field `@id` for numeric and grouping operations.**

In [ ]:
# Example: Filter by a numeric field, normalize, and group.

# Select a numeric field's @id and group field's @id.
# You can update these based on available columns.
# For demonstration, we auto-select the first float/int column for numeric_field, and next for group_field.
main_df = dataframes[main_record_set_id]

# Guess numeric columns
numeric_candidates = main_df.select_dtypes(include=['float', 'int']).columns.tolist()
if not numeric_candidates:
    # Try to cast columns to float
    for col in main_df.columns:
        try:
            pd.to_numeric(main_df[col])
            numeric_candidates.append(col)
        except Exception:
            continue

if len(numeric_candidates) == 0:
    raise ValueError("No numeric columns found.")

numeric_field = numeric_candidates[0]  # Use @id (column/field name in dataframe)
print(f"Using numeric field for analysis: '{numeric_field}'")

threshold = main_df[numeric_field].astype(float).mean() if main_df[numeric_field].dtype != float else 10

# Filter records
filtered_df = main_df[pd.to_numeric(main_df[numeric_field], errors='coerce') > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (count: {len(filtered_df)}):")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (pd.to_numeric(filtered_df[numeric_field]) - pd.to_numeric(filtered_df[numeric_field]).mean()) / pd.to_numeric(filtered_df[numeric_field]).std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to select a group field (categorical)
group_candidates = [col for col in main_df.columns if main_df[col].dtype == object and col != numeric_field]
group_field = group_candidates[0] if group_candidates else None

if group_field:
    print(f"\nGrouping by field: '{group_field}'")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(grouped_df.head())
else:
    print("No group field found for grouping operation.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and its relationship to the group field (if available). Uses matplotlib and seaborn where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(pd.to_numeric(main_df[numeric_field], errors='coerce').dropna(), bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field is available, visualize average by group
if group_field:
    plt.figure(figsize=(10,6))
    avg_vals = main_df.groupby(group_field)[numeric_field].mean(numeric_only=True)
    avg_vals = avg_vals.sort_values(ascending=False)[:20]  # Top 20 only
    sns.barplot(x=avg_vals.values, y=avg_vals.index)
    plt.title(f"Mean {numeric_field} by {group_field} (top 20)")
    plt.xlabel(f"Mean {numeric_field}")
    plt.ylabel(group_field)
    plt.show()

## 6. Conclusion
- Used `mlcroissant` to access Croissant schema and dataset metadata.
- Extracted protein records table (main record set) and identified fields by their `@id` values (DataFrame columns).
- Performed basic exploratory data filtering and normalization for a numeric field.
- Visualized distribution of protein numeric attributes and mean values by groupings.

This demonstrates how you can use Croissant schemas and mlcroissant to reproducibly explore and analyze complex scientific datasets.

#### Next Steps
- Consider more detailed field-level cleaning or feature engineering.
- Apply domain knowledge for downstream statistical or ML tasks.